# Preprocessing & Data Augmentation

Notebook ini akan mengimplementasikan tahapan praproses dan augmentasi data sesuai `notes.txt`, meliputi:
1. **(Opsional) Face Detection & Cropping:** Fungsi *standby* jika kedepannya ada gambar asli yang belum di-crop.
2. **Custom Preprocessing:** Grayscale Conversion, CLAHE, Gaussian Blur, Thresholding, Morphological Closing.
3. **Data Augmentation & Splitting:** Memanfaatkan `ImageDataGenerator` pada set Train (dengan parameter rotasi, shift, zoom, flip) dan preprocessing dasar pada Val/Test.
4. **Penyimpanan:** Menyimpan hasil akhir gambar praproses (dan augmentasinya) ke direktori lokal `data/results/`.

In [1]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import kagglehub
from pathlib import Path
import os
import shutil
from tqdm import tqdm
from tensorflow.keras.preprocessing.image import ImageDataGenerator, img_to_array, load_img

plt.rcParams['figure.figsize'] = (15, 5)

## 1. (Opsional) Face Detection & Cropping
Disediakan sebagai referensi jika sistem menerima foto mentah (bukan gambar wajah saja). Menggunakan Haar Cascade.

In [2]:
def detect_and_crop_face(image_path):
    """
    Fungsi opsional untuk mendeteksi dan meng-crop wajah menggunakan Haar Cascade.
    """
    face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
    img = cv2.imread(image_path)
    if img is None: return None
    
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(30, 30))
    
    if len(faces) == 0: return img  # Return original if no face detected
        
    x, y, w, h = faces[0]
    return img[y:y+h, x:x+w]

# Contoh pemanggilan (commented out):
# cropped_img = detect_and_crop_face('path/to/real_world_image.jpg')

## 2. Definisi Custom Preprocessing Pipeline
Fungsi ini akan mengubah citra ke Grayscale, memproses CLAHE, Gaussian Blur, Threshold, dan Morphological Closing.

In [3]:
def apply_custom_preprocessing(img_array):
    """
    Menerima numpy array gambar (H, W, 3), dan mengembalikan array gambar (H, W, 1) 
    setelah melalui pipeline grayscale, CLAHE, Blur, dan Morphologi.
    Fungsi ini kompatibel dengan `preprocessing_function` pada ImageDataGenerator.
    """
    # Pastikan array uint8
    img = img_array.astype(np.uint8)
    
    # Grayscale
    if len(img.shape) == 3 and img.shape[-1] == 3:
        img_gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    else:
        img_gray = img
        
    # 1. CLAHE
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    img_clahe = clahe.apply(img_gray)
    
    # 2. Gaussian Blur
    img_blur = cv2.GaussianBlur(img_clahe, (5, 5), 0)
    
    # 3. Operasi Morfologi (Otsu + Closing)
    _, img_thresh = cv2.threshold(img_blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    kernel = np.ones((5,5), np.uint8)
    img_morph = cv2.morphologyEx(img_thresh, cv2.MORPH_CLOSE, kernel)
    
    # Output final bisa blur atau morph. 
    # Di sini kita gunakan img_blur yang sudah di-CLAHE untuk fitur yang lebih halus (tidak biner)
    final_img = img_blur
    
    # Kembalikan ke shape (H, W, 1) agar kompatibel dengan Keras grayscale mode
    return np.expand_dims(final_img, axis=-1).astype(np.float32)


## 3. Direktori Setup (Input Kaggle & Output Lokal)

In [4]:
# Direktori Input (Kagglehub)
path = kagglehub.dataset_download("ashishjangra27/face-mask-12k-images-dataset")
input_dir = Path(path) / "Face Mask Dataset"
if not input_dir.exists():
    input_dir = Path(path)
    
# Direktori Output (Target)
RESULTS_ROOT = "../data/results" # Silakan ubah path ini sesuai keinginan
output_dir = Path(RESULTS_ROOT).resolve()

splits = ["Train", "Validation", "Test"]
classes = ["WithMask", "WithoutMask", "MaskWornIncorrect"]

print("Input:", input_dir)
print("Output:", output_dir)

Input: C:\Users\hp\.cache\kagglehub\datasets\ashishjangra27\face-mask-12k-images-dataset\versions\1\Face Mask Dataset
Output: D:\naufalarizq\ipb\kuliah\depart\sem-6\pcd\projek\Klasifikasi-Penggunaan-Masker-Pada-Wajah\data\results


## 4. Eksekusi Preprocessing & Augmentasi (Menyimpan ke Disk)
Kode di bawah ini akan menginisialisasi `ImageDataGenerator` dan mengekspor hasilnya langsung ke folder `data/results`. 
- **Validation & Test:** Hanya Resize, Preprocess, dan rescale=1./255.
- **Train:** Sama seperti di atas + Data Augmentation (Rotation, Shift, Zoom, Flip).

In [5]:
# Konfigurasi ImageGenerator sesuai catatan notes.txt
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=10,
    width_shift_range=0.2,
    height_shift_range=0.2,
    zoom_range=0.25,
    horizontal_flip=True,
    samplewise_center=True, 
    samplewise_std_normalization=True,
    fill_mode='nearest',
    preprocessing_function=apply_custom_preprocessing
)

test_datagen = ImageDataGenerator(
    rescale=1./255,
    preprocessing_function=apply_custom_preprocessing
)

TARGET_SIZE = (224, 224)
BATCH_SIZE = 32

In [6]:
def save_augmented_data(generator, input_split_dir, output_split_dir, augment_multiplier=1):
    """
    Membaca data dari input_split_dir menggunakan generator (misal: train_datagen), 
    lalu menyimpannya secara fisik ke output_split_dir.
    """
    if not input_split_dir.exists(): return
    
    if output_split_dir.exists():
        shutil.rmtree(output_split_dir)
    
    data_flow = generator.flow_from_directory(
        input_split_dir,
        target_size=TARGET_SIZE,
        color_mode='grayscale', # Memaksa load grayscale karena final kita array 1 channel
        batch_size=BATCH_SIZE,
        class_mode='categorical',
        shuffle=False,
        save_to_dir=str(output_split_dir),
        save_prefix='aug',
        save_format='png'
    )
    
    # Buat sub-folder target manual jika flow_from_directory belum membuatnya
    for cls in data_flow.class_indices.keys():
        (output_split_dir / cls).mkdir(parents=True, exist_ok=True)
        
    # flow_from_directory meletakkan semua output ke save_to_dir secara langsung tanpa subfolder class.
    # Kita modifikasi untuk menyimpan berurutan ke subfoldernya agar terstruktur:
    
    print(f"Memproses {input_split_dir.name}...")
    total_images = data_flow.samples
    batches = (total_images // BATCH_SIZE) + (1 if total_images % BATCH_SIZE > 0 else 0)
    
    # Jalankan iterasi untuk N multiplier (Train = N augments, Test = 1 pass)
    for _ in range(augment_multiplier):
        for i in tqdm(range(batches), desc="Batches"):
            # Ketika memanggil next(data_flow), Keras otomatis memproses dan menyimpannya (jika save_to_dir terisi)
            # Namun karena kita mau mengatur path per subfolder, kita tidak pakai save_to_dir bawaan flow.
            batch_x, batch_y = next(data_flow)
            
            # Simpan manual per batch ke folder yang sesuai dengan class
            start_idx = i * BATCH_SIZE
            for j in range(len(batch_x)):
                idx = start_idx + j
                # Cari class index
                cls_idx = np.argmax(batch_y[j])
                cls_name = list(data_flow.class_indices.keys())[cls_idx]
                
                # Image array adalah rescale 0..1 (atau distandarisasi), harus di-denormalisasi sblm disave dengan cv2
                # ATAU gunakan original array jika bisa. Karena kita rescale=1./255, kita kali 255.
                # Perlu hati-hati dengan samplewise_std_normalization, 
                # Jika disave ke disk untuk ML klasik, normalisasi sebaiknya dilakukan saat training saja, 
                # namun instruksi mengharuskan menyimpan results.
                img_to_save = batch_x[j] * 255.0
                # clip agar aman
                img_to_save = np.clip(img_to_save, 0, 255).astype(np.uint8)
                
                out_path = output_split_dir / cls_name / f"processed_{idx}_{_}.png"
                cv2.imwrite(str(out_path), img_to_save)

# Karena loop manual di atas lebih rapi dan menjamin sub-folder label tetap ada,
# kita hapus parameter save_to_dir di setup flow-nya agar tidak numpuk di root directory.


In [ ]:
def run_pipeline():
    # 1. Validation & Test (Tanpa augmentasi, multiplier = 1)
    for split in ["Validation", "Test"]:
        in_dir = input_dir / split
        out_dir = output_dir / "processed" / split
        
        test_flow = test_datagen.flow_from_directory(
            in_dir,
            target_size=TARGET_SIZE,
            color_mode='rgb', # Baca awal rgb, nanti diubah ke 1 channel di custom_preprocessing
            batch_size=BATCH_SIZE,
            class_mode='categorical',
            shuffle=False
        )
        
        # Create dirs
        for cls in test_flow.class_indices.keys():
            (out_dir / cls).mkdir(parents=True, exist_ok=True)
            
        print(f"\nMemproses {split}...")
        total_images = test_flow.samples
        batches = (total_images // BATCH_SIZE) + (1 if total_images % BATCH_SIZE > 0 else 0)
        
        for i in tqdm(range(batches), desc=f"Batches {split}"):
            batch_x, batch_y = next(test_flow)
            start_idx = i * BATCH_SIZE
            for j in range(len(batch_x)):
                cls_name = list(test_flow.class_indices.keys())[np.argmax(batch_y[j])]
                img_to_save = np.clip(batch_x[j] * 255.0, 0, 255).astype(np.uint8)
                cv2.imwrite(str(out_dir / cls_name / f"processed_{start_idx + j}.png"), img_to_save)

    # 2. Train (Dengan augmentasi, misalkan kita mau 2 kali ukuran original = multiplier 2)
    split = "Train"
    in_dir = input_dir / split
    out_dir = output_dir / "augmented" / split
    
    train_flow = train_datagen.flow_from_directory(
        in_dir,
        target_size=TARGET_SIZE,
        color_mode='rgb',
        batch_size=BATCH_SIZE,
        class_mode='categorical',
        shuffle=True # Shuffle saat training/augment
    )
    
    for cls in train_flow.class_indices.keys():
        (out_dir / cls).mkdir(parents=True, exist_ok=True)
        
    print(f"\nMemproses augmentasi pada {split}...")
    batches = (train_flow.samples // BATCH_SIZE) + (1 if train_flow.samples % BATCH_SIZE > 0 else 0)
    
    # Misal generate 2 set (1 set mirip original + 1 set fully aug)
    for aug_iter in range(2):
        for i in tqdm(range(batches), desc=f"Aug Iter {aug_iter+1}"):
            batch_x, batch_y = next(train_flow)
            for j in range(len(batch_x)):
                cls_name = list(train_flow.class_indices.keys())[np.argmax(batch_y[j])]
                
                # Keras samplewise_std_normalization membuat range jadi tidak [0, 1]. 
                # Jika disave ke gambar disk akan jadi masalah (harus 0-255 bulat).
                # Kita re-normalize ke [0, 255] untuk kepentingan penulisan ke disk image.
                img_vals = batch_x[j]
                img_vals = (img_vals - np.min(img_vals)) / (np.max(img_vals) - np.min(img_vals) + 1e-7)
                img_to_save = (img_vals * 255.0).astype(np.uint8)
                
                out_path = out_dir / cls_name / f"aug_{aug_iter}_{i*BATCH_SIZE+j}.png"
                cv2.imwrite(str(out_path), img_to_save)

# Uncomment untuk menjalankan semua pipeline pemrosesan data (membutuhkan waktu)
print("Fungsi run_pipeline() telah dideklarasikan.")

Fungsi run_pipeline() telah dideklarasikan.


In [8]:
run_pipeline()

Found 800 images belonging to 2 classes.

Memproses Validation...


Batches Validation: 100%|██████████| 25/25 [00:08<00:00,  2.79it/s]

Found 992 images belonging to 2 classes.



Memproses Test...


Batches Test: 100%|██████████| 31/31 [00:11<00:00,  2.68it/s]


Found 10000 images belonging to 2 classes.

Memproses augmentasi pada Train...


Aug Iter 2: 100%|██████████| 313/313 [03:50<00:00,  1.36it/s]
